# **Pipeline 1 - Data Ingestion**

In [ ]:
# Cell 1: Install specific library versions
!pip install pandas==1.5.3 \
                numpy==1.24.2 \
                sqlalchemy==2.0.8 \
                scikit-learn==1.2.2 \
                matplotlib==3.7.1 \
                seaborn==0.12.2 \
                gradio==4.44.1


In [ ]:
# Cell 2: Imports & styling
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sqlite3
from sqlalchemy import create_engine

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB, BernoulliNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA, QuadraticDiscriminantAnalysis as QDA
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.metrics import classification_report, roc_auc_score, roc_curve, accuracy_score

import gradio as gr
import seaborn as sns

# Use default matplotlib style for simplicity
plt.style.use('default')


In [ ]:
# Cell 3: Data Ingestion & preview
raw_df = pd.read_csv('USvideos.csv')
print("Raw data shape:", raw_df.shape)
raw_df.head(3)


In [ ]:
# Cell 4: Missing-value proportions (simple bar chart) 
# [DON'T TYPE] This cell is to show how many columns have missing value, so we can decide whether to fill them in, drop them, or leave them before preprocessing
missing = raw_df.isnull().mean() # Calculate the proportion of missing values for each column, the isnull method will return 0 or 1 for each row, then we use mean method to find the proportion
plt.bar(missing.index, missing.values)
plt.xticks(rotation=90)
plt.ylabel("Proportion Missing")
plt.title("Proportion Missing per Column")
plt.show()


In [ ]:
# Cell 5: Save data to SQL
engine = create_engine('sqlite:///youtube_trending.db') # Create a connection engine to a SQLite database file
raw_df.to_sql('raw_data', con=engine, if_exists='replace', index=False) # Save DataFrame as SQL table named 'raw_data'; replace if exists so you can rerun without running into error, and don’t include index or the first row since it's the header

conn = sqlite3.connect('youtube_trending.db') # Open a connection to the database
cnt = pd.read_sql_query("SELECT COUNT(*) AS cnt FROM raw_data", conn).iloc[0,0] # Run SQL query to count number of rows in the table
print("Rows in raw_data table:", cnt) # Display how many rows were successfully saved


# **Pipeline 2 - Data Preprocessing**

In [ ]:
# Cell 6: Data cleaning via SQL
c = conn.cursor() # Create a cursor object to execute SQL commands
c.execute("DROP TABLE IF EXISTS cleaned_data") # Drop table if exists, so you can rerun this cell without running into duplication error
c.execute("""
    CREATE TABLE cleaned_data AS  -- Create a new table named 'cleaned_data'
    SELECT *                      -- Copy all columns from raw_data
      FROM raw_data
     WHERE views > 0              -- Keep only rows where 'views' are positive
       AND video_error_or_removed = 0   -- Exclude videos that are unavailable or removed
       AND comments_disabled = 0        -- Exclude videos where comments are turned off
""")
conn.commit()  # Save (commit) the changes to the database

# Compare row counts before and after cleaning
before = pd.read_sql_query("SELECT COUNT(*) AS cnt FROM raw_data", conn).iloc[0,0]  # Count total rows before cleaning
after  = pd.read_sql_query("SELECT COUNT(*) AS cnt FROM cleaned_data", conn).iloc[0,0]  # Count total rows after cleaning
print(f"Before cleaning: {before} rows\nAfter cleaning:  {after} rows")  # Display the row counts before and after

# Quick overview of cleaned_data
cleaned_df = pd.read_sql_query("""
    SELECT views, likes, dislikes, comment_count  -- Select some important columns
      FROM cleaned_data                           -- From our cleaned SQL table
     LIMIT 100                                    -- Only take the first 100 rows for a quick sample
""", conn)
print("Sample of cleaned_data:")
display(cleaned_df)

stats = pd.read_sql_query("""
    SELECT
      AVG(views)         AS avg_views,
      AVG(likes)         AS avg_likes,
      AVG(dislikes)      AS avg_dislikes,
      AVG(comment_count) AS avg_comments,
      MIN(views)         AS min_views,
      MAX(views)         AS max_views
    FROM cleaned_data
""", conn)

pd.set_option("display.float_format", "{:.2f}".format)  # Round it to 2 decimal places
print("Numeric summary of cleaned_data:")
display(stats)


In [ ]:
# Cell 7: Views distribution before vs after cleaning (simple histograms)
before_v = pd.read_sql_query("SELECT views FROM raw_data", conn)['views']  # Get 'views' column from the original (uncleaned) dataset
after_v  = pd.read_sql_query("SELECT views FROM cleaned_data", conn)['views']  # Get 'views' column from the cleaned dataset

plt.hist(before_v, label='Before', range=(0, 1e6))  # Plot histogram of views before cleaning, limiting range to 0–1M views
plt.hist(after_v,  label='After',  range=(0, 1e6))  # Plot histogram of views after cleaning using same range
plt.xlabel("Views")
plt.ylabel("Count")
plt.title("Views Distribution: Before vs After Cleaning")
plt.legend()  # Show legend
plt.show()


# **Pipeline 3 - Data Exploration**

In [ ]:
# Cell 8: Weekly trending video counts (fix date parsing, simple line plot)
monthly = pd.read_sql_query("""
    SELECT
      date(  -- Convert the string date in 'trending_date' to a proper YYYY-MM-DD format
        '20' || substr(trending_date,1,2) || '-' ||  -- Year (e.g., '18' → '2018')
        substr(trending_date,7,2)       || '-' ||     -- Month (MM)
        substr(trending_date,4,2)                      -- Day (DD)
      ) AS TrendDate,
      COUNT(*) AS Count  -- Count how many videos trended on each date
    FROM cleaned_data
    GROUP BY TrendDate    -- Group by the converted date
    ORDER BY TrendDate    -- Sort results by date
""", conn)

monthly['TrendDate'] = pd.to_datetime(monthly['TrendDate'], format='%Y-%m-%d')  # Convert string to datetime object for resampling
weekly_counts = monthly.set_index('TrendDate')['Count'].resample('W').sum()     # Resample by week and sum up daily counts

# Visualize with plot
plt.plot(weekly_counts.index, weekly_counts.values, marker='o')  # Line plot showing weekly total trending videos
plt.xlabel("Week")
plt.ylabel("Number of Trending Videos")
plt.title("Weekly Trending Video Counts")
plt.show()  # Display the line plot

# Display raw daily count data
display(monthly)


# **Pipeline 4 - Feature Engineering**

In [ ]:
# Cell 9: Feature Engineering – Create & Visualize Basic Features

# Compute median views from cleaned_data
median_views = pd.read_sql_query("SELECT views FROM cleaned_data", conn)["views"].median()

# Calculate and create new features: total engagement (sum of views + likes + comments), title length, and popularity flag
c.execute("DROP TABLE IF EXISTS feat_basic")
c.execute(f"""
    CREATE TABLE feat_basic AS
    SELECT
        *,
        (views + likes + comment_count) AS Engagement,
        LENGTH(title)                   AS TitleLength,
        CASE
            WHEN views > {median_views:.0f} THEN 1
            ELSE 0
        END AS is_popular
    FROM cleaned_data
""")
conn.commit()

# Load new features from database
fb = pd.read_sql_query("SELECT Engagement, TitleLength, is_popular FROM feat_basic", conn)

# Plot Engagement distribution
# [DON'T TYPE] Visualize how engagement (views + likes + comments) is distributed among videos
plt.hist(fb['Engagement'])
plt.title("Engagement Distribution")
plt.xlabel("Engagement")
plt.ylabel("Count")
plt.show()

# Plot Title Length distribution
# [DON'T TYPE] Visualize how long the video titles are across the dataset
plt.hist(fb['TitleLength'])
plt.title("Video Title Length Distribution")
plt.xlabel("Title Length (chars)")
plt.ylabel("Count")
plt.show()

# Plot Popular vs Unpopular count
# [DON'T TYPE] Count how many videos are marked as popular (1) vs unpopular (0)
# [DON'T TYPE] Since the split is based on the median views, the two groups will have roughly equal sizes
counts = fb['is_popular'].value_counts().sort_index()
plt.bar(['Unpopular','Popular'], counts.values)
plt.title("Popularity Class Balance")
plt.ylabel("Count")
plt.show()

# Show the SQL table
display(fb)


In [ ]:
# Cell 10: Feature Engineering – like-to-view and comment-to-view ratios

# Calculate and create new features: like-to-view and comment-to-view ratios
c.execute("DROP TABLE IF EXISTS feat_ratios")
c.execute("""
    CREATE TABLE feat_ratios AS
    SELECT *,
           CAST(likes AS FLOAT)/views         AS like_ratio,
           CAST(comment_count AS FLOAT)/views AS comment_ratio
      FROM feat_basic
""")
conn.commit()

# Load the new features from database
fr = pd.read_sql_query("SELECT like_ratio, comment_ratio FROM feat_ratios", conn)

# Plot Like Ratio
plt.hist(fr['like_ratio'])
plt.title("Like-to-View Ratio")
plt.xlabel("Like Ratio")
plt.ylabel("Count")
plt.show()

# Plot Comment Ratio
plt.hist(fr['comment_ratio'])
plt.title("Comment-to-View Ratio")
plt.xlabel("Comment Ratio")
plt.ylabel("Count")
plt.show()

# Show the SQL table so far
display(fr)

In [ ]:
# Cell 11: Feature Engineering – Temporal features

# Calculate and create new features: like-to-view and comment-to-view ratios
c.execute("DROP TABLE IF EXISTS feat_temp")
c.execute("""
    CREATE TABLE feat_temp AS
    SELECT *,
           CAST(strftime('%H', publish_time) AS INTEGER) AS publish_hour,
           CAST(strftime('%w', publish_time) AS INTEGER) AS publish_dayofweek
      FROM feat_ratios
""")
conn.commit()

# Load the new features from database
ft = pd.read_sql_query("SELECT publish_hour, publish_dayofweek FROM feat_temp", conn)

# Plot Publish Hour
plt.hist(ft['publish_hour'], bins=24)
plt.title("Distribution of Publish Hour")
plt.xlabel("Hour of Day")
plt.ylabel("Count")
plt.show()

# Plot Publish Day of Week
plt.hist(ft['publish_dayofweek'], bins=7)
plt.title("Distribution of Publish Day")
plt.xlabel("Day of Week (0=Sun)")
plt.ylabel("Count")
plt.show()

# Show the SQL table so far
display(ft)


In [ ]:
# [CAN I REMOVE THIS?] New Cell: Histogram of the new ratio features
ratios_df = pd.read_sql_query("""
    SELECT like_ratio, comment_ratio
      FROM feat_ratios
    LIMIT 10000
""", conn)

plt.figure(figsize=(10,3))
plt.subplot(1,2,1)
plt.hist(ratios_df['like_ratio'].clip(upper=0.1), bins=30)
plt.title("Like-to-View Ratio (clipped at 0.1)")
plt.xlabel("like_ratio")
plt.ylabel("Count")

plt.subplot(1,2,2)
plt.hist(ratios_df['comment_ratio'].clip(upper=0.01), bins=30)
plt.title("Comment-to-View Ratio (clipped at 0.01)")
plt.xlabel("comment_ratio")
plt.tight_layout()
plt.show()

# Show the SQL table so far
display(ratios_df)


In [ ]:
# Cell 12: Feature Engineering – Channel aggregates & correlation
# ---------------------------------------------------------------
c.execute("DROP TABLE IF EXISTS channel_agg")
c.execute("""
    CREATE TABLE channel_agg AS
    SELECT channel_title,
           COUNT(*)    AS num_videos,
           SUM(views)  AS total_views,
           AVG(likes)  AS avg_likes
      FROM feat_temp
  GROUP BY channel_title
""")
conn.commit()

c.execute("DROP TABLE IF EXISTS features_final")
c.execute("""
    CREATE TABLE features_final AS
    SELECT t.*, c.num_videos, c.total_views, c.avg_likes
      FROM feat_temp t
 LEFT JOIN channel_agg c USING(channel_title)
""")
conn.commit()

ff = pd.read_sql_query("SELECT num_videos, total_views, avg_likes FROM features_final", conn)
corr = ff.corr()

# Simple heatmap
plt.figure(figsize=(5,4))
plt.imshow(corr, cmap='viridis', aspect='auto')
plt.colorbar(label='Correlation')
plt.xticks(range(len(corr)), corr.columns, rotation=45, ha='right')
plt.yticks(range(len(corr)), corr.columns)
plt.title("Channel-Level Feature Correlations")
plt.tight_layout()
plt.show()

# Show the SQL table so far
display(ff)


# **Pipeline 5 - Model Selection**

In [ ]:
# Cell 13: Prepare Data for Engagement‐Level Prediction & Extended EDA
# --------------------------------------------------------------------
# 1) Load our engineered feature table
df_feat = pd.read_sql_query("SELECT * FROM features_final", conn)

# 2) Compute median Engagement (views + likes + comments)
median_eng = df_feat['Engagement'].median()
print(f"Median Engagement: {median_eng:.0f}")

# 3) Define binary target: high engagement if above median
df_feat['high_engagement'] = (df_feat['Engagement'] > median_eng).astype(int)

# 4) Select input features (omit raw counts to avoid leakage)
features = [
    'TitleLength','like_ratio','comment_ratio',
    'publish_hour','publish_dayofweek',
    'num_videos','total_views','avg_likes'
]
X = df_feat[features]
y = df_feat['high_engagement']

# 5) Train/test split (stratify to keep class balance)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train shape:", X_train.shape, "Test shape:", X_test.shape)

# --- Extended Visualizations ---

# A) Class balance in training set
plt.figure(figsize=(4,3))
counts = y_train.value_counts().sort_index()
plt.bar(['Low','High'], counts.values, color=['skyblue','salmon'])
plt.title("Training Set: Low vs High Engagement")
plt.ylabel("Number of Videos")
plt.tight_layout()
plt.show()

# B) Title length distribution by engagement class
plt.figure(figsize=(6,3))
plt.hist(df_feat.loc[df_feat.high_engagement==0, 'TitleLength'],
         bins=30, alpha=0.5, label='Low Engagement')
plt.hist(df_feat.loc[df_feat.high_engagement==1, 'TitleLength'],
         bins=30, alpha=0.5, label='High Engagement')
plt.xlabel("Title Length (chars)")
plt.ylabel("Count")
plt.title("Title Length by Engagement Class")
plt.legend()
plt.tight_layout()
plt.show()

# C) Like-to-view ratio by engagement class
plt.figure(figsize=(6,3))
plt.hist(df_feat.loc[df_feat.high_engagement==0, 'like_ratio'],
         bins=30, alpha=0.5, label='Low Engagement')
plt.hist(df_feat.loc[df_feat.high_engagement==1, 'like_ratio'],
         bins=30, alpha=0.5, label='High Engagement')
plt.xlabel("Like-to-View Ratio")
plt.ylabel("Count")
plt.title("Like Ratio by Engagement Class")
plt.legend()
plt.tight_layout()
plt.show()

# D) Comment-to-view ratio by engagement class
plt.figure(figsize=(6,3))
plt.hist(df_feat.loc[df_feat.high_engagement==0, 'comment_ratio'],
         bins=30, alpha=0.5, label='Low Engagement')
plt.hist(df_feat.loc[df_feat.high_engagement==1, 'comment_ratio'],
         bins=30, alpha=0.5, label='High Engagement')
plt.xlabel("Comment-to-View Ratio")
plt.ylabel("Count")
plt.title("Comment Ratio by Engagement Class")
plt.legend()
plt.tight_layout()
plt.show()


# **Pipeline 6 - Hyperparameter Tuning**

In [ ]:
# Cell 14: Faster Hyperparameter Tuning & Evaluation
# -------------------------------------------------
# To cut training time, we:
#  - Reduce CV folds from 5 → 3
#  - Narrow each model’s grid to one or two settings
#  - Keep n_jobs=-1 so each search still uses all cores efficiently

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

# 1) Define our ensemble models
models = {
    'RandomForest':    RandomForestClassifier(random_state=42),
    'GradientBoosting':GradientBoostingClassifier(random_state=42),
    'AdaBoost':        AdaBoostClassifier(random_state=42)
}

# 2) Slimmed-down grids for quick tuning
param_grid = {
    'RandomForest': {
        'n_estimators': [100],    # single setting
        'max_depth':    [None, 10]
    },
    'GradientBoosting': {
        'n_estimators':   [100],
        'learning_rate':  [0.1]   # single setting
    },
    'AdaBoost': {
        'n_estimators':  [50],    # single setting
        'learning_rate': [1.0]
    }
}

tuned_acc = {}
tuned_models = {}

# 3) Run 3-fold CV for each model
for name, model in models.items():
    print(f"Tuning {name} (fast)…")
    gs = GridSearchCV(
        estimator=model,
        param_grid=param_grid[name],
        cv=3,              # fewer folds
        scoring='accuracy',
        n_jobs=-1
    )
    gs.fit(X_train, y_train)    # much faster grid search
    best = gs.best_estimator_
    tuned_models[name] = best

    # 4) Evaluate on test set
    preds = best.predict(X_test)
    tuned_acc[name] = accuracy_score(y_test, preds)
    print(f"→ {name} best params: {gs.best_params_}, test accuracy: {tuned_acc[name]:.4f}")


# **Pipeline 7 - Model Training and Evaluation**

In [ ]:
# Cell 15: Plot Tuned Model Accuracies
names      = list(tuned_acc.keys())
accuracies = [tuned_acc[n] for n in names]
x          = np.arange(len(names))

plt.figure(figsize=(6,4))
plt.bar(x, accuracies)
plt.xticks(x, names, rotation=45, ha='right')
plt.ylim(0,1)
plt.ylabel("Accuracy")
plt.title("Tuned Ensemble Model Accuracies for Engagement")
plt.tight_layout()
plt.show()


# **Pipeline 8 - Model Deployment**

In [ ]:
# Cell 16: Gradio Dashboard — Title Length EDA & Extended Top Channels
def show_dashboard():
    # 1) Title length distribution by engagement class
    fig1, ax1 = plt.subplots(figsize=(6,3))
    ax1.hist(df_feat.loc[df_feat.high_engagement==0, 'TitleLength'],
             bins=30, alpha=0.5, label='Low Engagement')
    ax1.hist(df_feat.loc[df_feat.high_engagement==1, 'TitleLength'],
             bins=30, alpha=0.5, label='High Engagement')
    ax1.set_xlabel("Title Length (chars)")
    ax1.set_ylabel("Count")
    ax1.set_title("Title Length by Engagement Class")
    ax1.legend()
    fig1.tight_layout()

    # 2) Tuned model accuracies
    fig2, ax2 = plt.subplots(figsize=(6,3))
    names_list = list(tuned_acc.keys())
    accs = [tuned_acc[n] for n in names_list]
    x = np.arange(len(names_list))
    ax2.bar(x, accs, color='lightgreen')
    ax2.set_xticks(x)
    ax2.set_xticklabels(names_list, rotation=45, ha='right')
    ax2.set_ylim(0,1)
    ax2.set_ylabel("Accuracy")
    ax2.set_title("Tuned Model Accuracies")
    fig2.tight_layout()

    # 3) Top-10 channels by average predicted engagement
    #    using the best RandomForest model
    best_model = tuned_models['RandomForest']
    probs = best_model.predict_proba(X_test)[:,1]
    video_meta = df_feat.loc[X_test.index, ['channel_title']].copy()
    video_meta['pred_high_eng'] = probs
    channel_probs = video_meta.groupby('channel_title')['pred_high_eng'].mean()
    top10_channels = (
        channel_probs
        .sort_values(ascending=False)
        .head(10)
        .reset_index()
        .rename(columns={'pred_high_eng': 'avg_pred_high_eng'})
    )

    # 4) Feature importances of the RandomForest
    importances = best_model.feature_importances_
    imp_df = pd.Series(importances, index=features).sort_values()
    fig4, ax4 = plt.subplots(figsize=(6,3))
    imp_df.plot.barh(ax=ax4)
    ax4.set_title("RandomForest Feature Importances")
    fig4.tight_layout()

    # Return: title-length hist, accuracies bar, top10 DataFrame, importances
    return fig1, fig2, top10_channels, fig4

gr.Interface(
    fn=show_dashboard,
    inputs=[],
    outputs=[
        gr.Plot(label="Title Length by Engagement Class"),
        gr.Plot(label="Tuned Model Accuracies"),
        gr.Dataframe(label="Top 10 Channels by Predicted Engagement"),
        gr.Plot(label="Feature Importances")
    ],
    title="US YouTube Video Engagement Dashboard & Model Insights",
    description="Title length EDA, tuned model performance, top channels, and feature importances."
).launch()
